# Fantasy Premier League points prediction

One pass from raw gameweek files to trained models, in the order it runs.

| stage | produces |
|---|---|
| 1. Load and merge | `all_seasons_data.csv` |
| 2. Match index | `game_number` |
| 3. Features | `all_seasons_data_featured.csv` |
| 4. Train | `saved_models/`, `model_metrics.json` |
| 5. Two-stage models | `P(plays)`, `E[points\|plays]`, `P(haul)` |
| 6. Save | artifacts for `scripts/predict_gameweek.py` |

The `scripts/` directory runs these same cells headlessly, one stage per
script, so this notebook stays the single definition of the pipeline.

Two environment switches:

- `FPL_FEATURE_SET=full` trains on all features instead of the compact set
- `FPL_SPLIT=xg` confines train/val/test to the seasons that record xG

## 1. Load and merge every season

Reads each season's merged_gw.csv, maps positions, teams and opponents to
names, harmonises columns that differ between seasons, and re-scores
2016-17 to 2018-19 under the current rules.

### What differs between seasons

Ten years of FPL exports do not share a schema. Rather than one hardcoded statement per season, the differences are listed as data and applied by a single loop.

In [ ]:
# ---------------------------------------------------------------------------
# What differs between seasons
# ---------------------------------------------------------------------------
# FPL's export has changed shape repeatedly over ten years: columns appear,
# disappear, and are renamed. Every season therefore needs slightly different
# handling before the ten can be concatenated.
#
# Collecting those differences here, as data, keeps the processing below as a
# single loop. The alternative -- one hardcoded statement per season, which is
# how this started -- is twenty near-identical lines that have to be edited
# every August, and where a missed season fails silently.

SEASONS = [
    '2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
    '2021-22', '2022-23', '2023-24', '2024-25', '2025-26',
]

# The early exports carry a wide tail of columns that later seasons dropped.
# Keeping them would mean nine seasons of NaN for the sake of three.
LEGACY_ONLY_COLUMNS = [
    'attempted_passes', 'big_chances_created', 'big_chances_missed',
    'completed_passes', 'dribbles', 'ea_index', 'errors_leading_to_goal',
    'errors_leading_to_goal_attempt', 'fouls', 'key_passes',
    'kickoff_time_formatted', 'loaned_in', 'loaned_out', 'offside',
    'open_play_crosses', 'penalties_conceded', 'tackled', 'target_missed',
    'winning_goals', 'id',
]

# Expected goals and `starts` begin in 2022-23. They are dropped here so the
# ten seasons share a schema, then joined back on player and fixture ids by
# scripts/build_dataset.py, which can flag the seasons that never had them.
MODERN_ONLY_COLUMNS = [
    'expected_goals', 'expected_assists', 'expected_goals_conceded',
    'expected_goal_involvements', 'starts',
]

# Season-specific quirks.
DROP_XP_FROM = '2020-21'          # xP exists from here on; not comparable, so dropped
LEGACY_SEASONS = ['2016-17', '2017-18', '2018-19']
MODERN_SEASONS = ['2022-23', '2023-24', '2024-25', '2025-26']
DROP_MODIFIED = ['2024-25', '2025-26']

# position and team are absent from the early exports and have to be looked up
# from players_raw.csv and master_team_list.csv.
NEEDS_POSITION_LOOKUP = ['2016-17', '2017-18', '2018-19', '2019-20']

# FPL only began recording tackles, recoveries and clearances per match in
# 2025-26. For the seasons in between they are filled with zero here and
# replaced with real FBref values later; the has_fbref_defensive flag records
# which rows are which, so a model can tell a genuine zero from a missing one.
NEEDS_DEFENSIVE_PLACEHOLDERS = [
    '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25',
]
DEFENSIVE_PLACEHOLDER_COLUMNS = [
    'clearances_blocks_interceptions', 'recoveries', 'tackles',
]

# Defensive contribution is scored for every season that has the inputs.
# 2025-26 ships the figure itself, so it is left alone.
SCORE_DEFENSIVE_CONTRIBUTION = SEASONS[:-1]

# The 2025-26 rules award 2 points for reaching a defensive-contribution
# threshold. Seasons played under the old rules are re-scored so that ten
# years of target values mean the same thing. 2019-20 to 2024-25 are re-scored
# later instead, once their real FBref numbers have been merged in -- doing it
# here would score them against placeholder zeros.
RESCORE_POINTS = LEGACY_SEASONS

ELEMENT_TYPE_TO_POSITION = {
    1: 'Goalkeeper', 2: 'Defender', 3: 'Midfielder', 4: 'Forward',
}
POSITION_CODES = {
    'Goalkeeper': 'GK', 'Defender': 'DEF', 'Midfielder': 'MID', 'Forward': 'FWD',
}

print(f"{len(SEASONS)} seasons configured: {SEASONS[0]} to {SEASONS[-1]}")

### Load, align and merge

One pass per season: read it, fill in anything the export omits, drop the columns that do not exist in all ten, score defensive contribution, and re-score the seasons played under the old rules so that every row's `total_points` means the same thing.

In [ ]:
import os


def read_season_csv(path):
    """Read one of FPL's exports without guessing at its encoding.

    The archive is a mix: some seasons are utf-8, others latin-1, and a few
    rows in the older files have stray delimiters that no encoding fixes.
    """
    for encoding in ('utf-8', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=encoding, on_bad_lines='skip',
                               low_memory=False)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"could not decode {path}")


def load_position_and_team(gw, season, master_team_list):
    """Fill in position and team for the seasons that do not ship them.

    From 2020-21 the gameweek export carries both. Before that it carries only
    `element`, the player id, so both have to come from players_raw.csv --
    element_type for the position, and team id resolved through the master list
    for the club name.
    """
    players_raw = read_season_csv(os.path.join('data', season, 'players_raw.csv'))

    position_by_player = dict(zip(
        players_raw['id'],
        players_raw['element_type'].map(ELEMENT_TYPE_TO_POSITION),
    ))
    gw['position'] = gw['element'].map(position_by_player)

    season_teams = master_team_list[master_team_list['season'] == season]
    name_by_team_id = dict(zip(season_teams['team'], season_teams['team_name']))
    team_by_player = {
        player_id: name_by_team_id.get(team_id, 'Unknown')
        for player_id, team_id in zip(players_raw['id'], players_raw['team'])
    }
    gw['team'] = gw['element'].map(team_by_player)
    return gw


def defensive_contribution(row):
    """FPL's defensive-contribution stat, by position.

    Defenders are credited for clearances, blocks, interceptions and tackles.
    Midfielders and forwards additionally get ball recoveries. Goalkeepers do
    not score it at all.
    """
    clearances = row.get('clearances_blocks_interceptions', 0)
    tackles = row.get('tackles', 0)
    recoveries = row.get('recoveries', 0)

    if row['position'] == 'Defender':
        return clearances + tackles
    if row['position'] in ('Midfielder', 'Forward'):
        return clearances + tackles + recoveries
    return 0


def rescore_under_current_rules(row):
    """Re-award the 2025-26 defensive-contribution bonus retrospectively.

    Without this, a defender's 6 points in 2017-18 and the same performance in
    2025-26 carry different target values, and the model is asked to fit two
    different scoring systems at once.
    """
    points = row['total_points']
    contribution = row['defensive_contribution']

    if row['position'] == 'Defender' and contribution >= 10:
        return points + 2
    if row['position'] in ('Midfielder', 'Forward') and contribution >= 12:
        return points + 2
    return points


master_team_list = read_season_csv('data/master_team_list.csv')

frames = []
for season in SEASONS:
    gw = read_season_csv(os.path.join('data', season, 'gws', 'merged_gw.csv'))
    started_with = gw.shape[1]

    if season in NEEDS_POSITION_LOOKUP:
        gw = load_position_and_team(gw, season, master_team_list)

    # Bring the schema into line with the other nine seasons.
    drop = []
    if season >= DROP_XP_FROM:
        drop.append('xP')
    if season in LEGACY_SEASONS:
        drop += LEGACY_ONLY_COLUMNS
    if season in MODERN_SEASONS:
        drop += MODERN_ONLY_COLUMNS
    if season in DROP_MODIFIED:
        drop.append('modified')
    gw = gw.drop(columns=[c for c in drop if c in gw.columns])

    if season in NEEDS_DEFENSIVE_PLACEHOLDERS:
        for column in DEFENSIVE_PLACEHOLDER_COLUMNS:
            gw[column] = 0

    if season in SCORE_DEFENSIVE_CONTRIBUTION:
        gw['defensive_contribution'] = gw.apply(defensive_contribution, axis=1)

    if season in RESCORE_POINTS:
        gw['total_points'] = gw.apply(rescore_under_current_rules, axis=1)

    gw['season'] = season
    frames.append(gw)
    print(f"  {season}  {len(gw):>7,} rows  {started_with:>3} -> {gw.shape[1]:>2} columns")

# pd.concat unions columns and fills the gaps with NaN, so a season with one
# column too many is not an error -- it silently becomes a column that is null
# for the other nine. Checking first turns that into a failure with a name
# attached.
schemas = {season: set(frame.columns) for season, frame in zip(SEASONS, frames)}
shared = set.intersection(*schemas.values())
mismatched = {s: sorted(cols - shared) for s, cols in schemas.items() if cols != shared}
if mismatched:
    raise ValueError(
        "seasons do not share a schema, so concatenating them would create "
        f"silently-null columns:\n  " +
        "\n  ".join(f"{s}: extra {cols}" for s, cols in mismatched.items())
    )
print(f"\nschema check: all {len(SEASONS)} seasons share {len(shared)} columns")

all_seasons_data = pd.concat(frames, ignore_index=True)
all_seasons_data['position'] = all_seasons_data['position'].replace(POSITION_CODES)

print(f"merged: {len(all_seasons_data):,} rows x {all_seasons_data.shape[1]} columns")

### Opponent names

`opponent_team` arrives as a numeric id whose meaning changes every season. The features join on the name, so it has to be resolved here.

In [ ]:
# Convert opponent_team from team IDs to team names using master team list
#
# data/master_team_list.csv only covers 2016-17..2023-24. For 2024-25 and
# 2025-26 the season lookup came back empty and `.get(team_id, team_id)` fell
# through to the raw numeric id, leaving opponent_team as "1", "2", ... for
# those two seasons.
#
# That was not a cosmetic problem. add_opponent_strength_features() joins
# team-level aggregates on the opponent NAME, so all 15 opponent-strength and
# advantage features came out NaN for 2024-25 and 2025-26 -- and the later
# dropna(subset=training_features) then deleted every row of both seasons.
# The two most recent seasons were silently absent from training entirely.
#
# Each season ships data/<season>/teams.csv with the same id -> name mapping,
# so those fill the gap.

import os

team_id_name_mapping = {}
for season in all_seasons_data['season'].unique():
    season_team_data = master_team_list[master_team_list['season'] == season]
    mapping = dict(zip(season_team_data['team'], season_team_data['team_name']))

    if not mapping:
        teams_path = os.path.join('data', season, 'teams.csv')
        if os.path.exists(teams_path):
            teams_df = read_season_csv(teams_path)
            mapping = dict(zip(teams_df['id'], teams_df['name']))
            print(f"  {season}: not in master_team_list, "
                  f"used {teams_path} ({len(mapping)} teams)")
        else:
            print(f"  WARNING: {season} has no team mapping and no {teams_path}")
    team_id_name_mapping[season] = mapping


# replace the opponent_team in all_seasons_data based on the season and the team id
def replace_opponent_team(row):
    season = row['season']
    team_id = row['opponent_team']
    return team_id_name_mapping[season].get(team_id, team_id)


all_seasons_data['opponent_team'] = all_seasons_data.apply(replace_opponent_team, axis=1)

# Nothing downstream works if an id survives here, so check rather than assume.
unmapped = (
    all_seasons_data['opponent_team']
    .astype(str)
    .str.fullmatch(r'\d+(\.\d+)?')
)
if unmapped.any():
    offending = (
        all_seasons_data.loc[unmapped]
        .groupby('season')
        .size()
        .to_dict()
    )
    raise ValueError(
        f"opponent_team is still a numeric id for {int(unmapped.sum()):,} rows: "
        f"{offending}.\n"
        f"  Opponent-strength features join on the team name, so these rows "
        f"would end up all-NaN and be dropped from training."
    )

print(f"opponent_team mapped to names for all {len(all_seasons_data):,} rows")
print(f"  seasons covered: {sorted(team_id_name_mapping)}")

### Checkpoint

`all_seasons_data.csv` is the merged, schema-aligned dataset. `scripts/build_dataset.py` picks it up from here to add the FBref defensive columns and expected goals.

In [ ]:
# save all seasons data to a csv file
# utf-8, matching cell 47 and every reader below. This used to write
# latin-1, which only worked because cell 47 rewrote the file as utf-8
# before anything read it -- and the pipeline scripts skip cell 47.
all_seasons_data.to_csv('all_seasons_data.csv', index=False, encoding='utf-8')

## 2. Chronological match index

game_number is each player's match number within a season, assigned from
kickoff time. Every lag and rolling feature sorts on it, and the models
drop rows below game_number 5 because their history is too short.

In [ ]:
# Assign game_number using kickoff_time for ALL seasons

print("="*80)
print("Assigning game_number using kickoff_time approach")
print("="*80)

# Parse kickoff_time to datetime for sorting
print("\nParsing kickoff_time...")
all_seasons_updated = all_seasons_df.copy()
all_seasons_updated['kickoff_datetime'] = pd.to_datetime(all_seasons_updated['kickoff_time'])

# Sort by player, season, and kickoff_time (chronological order)
print("Sorting records by player, season, and kickoff_time...")
all_seasons_updated = all_seasons_updated.sort_values(['element', 'season', 'kickoff_datetime'])

# Assign game_number for each player-season combination
print("Assigning game_number...")
all_seasons_updated['game_number'] = (
    all_seasons_updated
    .groupby(['element', 'season'])
    .cumcount() + 1
)

# Convert to nullable integer
all_seasons_updated['game_number'] = all_seasons_updated['game_number'].astype('Int64')

# Clean up temporary column
all_seasons_updated = all_seasons_updated.drop('kickoff_datetime', axis=1)

# Verify results
print(f"\n SUCCESS: game_number assigned to all records!")
print(f"   Total records: {len(all_seasons_updated):,}")
print(f"   Records with game_number: {all_seasons_updated['game_number'].notna().sum():,}")
print(f"   Records missing game_number: {all_seasons_updated['game_number'].isna().sum():,}")

# Show distribution
print(f"\nGame number statistics:")
print(f"   Min: {all_seasons_updated['game_number'].min()}")
print(f"   Max: {all_seasons_updated['game_number'].max()}")
print(f"   Mean: {all_seasons_updated['game_number'].mean():.1f}")

# Sample verification
print("\nSample verification - Mohamed Salah 2017-18 (first 5 games):")
salah_sample = all_seasons_updated[
    (all_seasons_updated['name'].str.contains('Salah', na=False)) &
    (all_seasons_updated['season'] == '2017-18')
][['name', 'game_number', 'kickoff_time', 'opponent_team', 'GW']].head(5)
print(salah_sample.to_string(index=False))

print("\n" + "="*80)

## 3. Feature engineering

Everything a model sees has to be knowable *before* the match it is predicting.
That single constraint shapes this whole section: every feature is a lag or a
rolling mean of matches already played, shifted one fixture inside the player
group.

Six families, in the order they are built:

| family | prefix | what it captures |
|---|---|---|
| previous-match statistics | `*_prev_1..5` | the last five matches, stat by stat |
| rolling player form | `*_rolling_3/5/10` | short and long-run form |
| season and price context | `is_home`, `price_*` | where in the season, and price momentum |
| availability | `avail_*` | whether the player will be on the pitch at all |
| expected goals | `xg_*` | chances created, rather than whether they went in |
| fixture difficulty | `fx_*` | this team's attack against that defence, by venue |

An earlier opponent-strength family was removed here. It mixed whole-season
records with recent form, ignored home and away, and measured a team's
aggregate points rather than the goals a player needs. Ablation put it at
-0.009 R2 on its own and +0.0002 on top of everything else: seventeen columns
carrying nothing. `fx_*` replaces it and is worth +0.0074.

In [ ]:
# Written with encoding='latin-1' by older versions of this pipeline;
# pandas defaults to utf-8 on read, so fall back rather than crash on the
# first accented player name.
try:
    all_seasons_data = pd.read_csv('all_seasons_data_final.csv')
except UnicodeDecodeError:
    all_seasons_data = pd.read_csv('all_seasons_data_final.csv', encoding='latin-1')

In [ ]:
# Create my_team_score and opponent_team_score columns based on was_home
all_seasons_data['my_team_score'] = all_seasons_data.apply(
    lambda row: row['team_h_score'] if row['was_home'] else row['team_a_score'],
    axis=1
)
all_seasons_data['opponent_team_score'] = all_seasons_data.apply(
    lambda row: row['team_a_score'] if row['was_home'] else row['team_h_score'],
    axis=1
)

In [ ]:
# Create result indicator: 1 = win, 0 = draw, -1 = loss
all_seasons_data['result'] = all_seasons_data.apply(
    lambda row: 1 if row['my_team_score'] > row['opponent_team_score']
                else (0 if row['my_team_score'] == row['opponent_team_score'] else -1),
    axis=1
)

### Previous-match statistics

The last five values of every match statistic, per player. `shift` inside the
player group is what keeps the current match out of its own features.

Cross-season carry-over is off by default: a player's form in May says little
about his form the following August, after a summer, a transfer window and
possibly a new club.

In [ ]:
def add_previous_game_stats(df, n_gameweeks=5, use_cross_season=False):

    # Columns to exclude from previous gameweek calculation
    exclude_columns = ['name', 'element', 'GW', 'game_number', 'position', 'team',
                       'season', 'fixture', 'kickoff_time', 'round', 'team_h_score', 'team_a_score']

    # Get stat columns (only numeric columns except the excluded ones)
    stat_columns = [col for col in df.columns
                    if col not in exclude_columns and pd.api.types.is_numeric_dtype(df[col])]

    # Determine groupby columns and sort order based on cross_season parameter
    if use_cross_season:
        # Use 'name' for cross-season continuity (element IDs change between seasons)
        # Sort by name, season, game_number for correct chronological order
        df_sorted = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)
        group_cols = ['name']
    else:
        # Group by element and season - stats only within same season
        df_sorted = df.sort_values(['element', 'season', 'game_number']).reset_index(drop=True)
        group_cols = ['element', 'season']

    # Create a copy to avoid modifying the original
    df_result = df_sorted.copy()

    # For each stat column, create N previous game columns
    for stat in stat_columns:
        for i in range(1, n_gameweeks + 1):
            col_name = f'{stat}_prev_{i}'
            # Shift by i positions for each player (based on game_number order)
            df_result[col_name] = df_result.groupby(group_cols, sort=False)[stat].shift(i)

    return df_result

### Rolling form

Means over the last 3, 5 and 10 matches. Three windows rather than one because
they answer different questions -- a 3-match mean catches a hot streak, a
10-match mean describes the player.

In [ ]:
def add_rolling_player_stats(df, windows=[3, 5, 10]):

    # Key performance metrics to calculate rolling averages for
    rolling_stats = ['total_points', 'goals_scored', 'assists', 'minutes',
                     'bonus', 'bps', 'clean_sheets', 'saves',
                     'ict_index', 'creativity', 'threat', 'influence']

    # Sort by player NAME (consistent across seasons), then season and game_number
    df = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)

    # Calculate rolling averages for each window size
    for window in windows:
        for stat in rolling_stats:
            if stat in df.columns:
                col_name = f'{stat}_rolling_{window}'
                # Calculate rolling mean using transform() with shift INSIDE the group
                # This ensures shift only happens within each player's data
                df[col_name] = (
                    df.groupby('name')[stat]
                    .transform(lambda x: x.rolling(window=window, min_periods=1).mean().shift(1))
                )

    return df

### Season and price context

Home advantage, how far into the season it is, and price momentum. Price moves
when the crowd moves, so a rising price is a weak proxy for information the
model does not otherwise have.

In [ ]:
def add_context_features(df):

    # 1. Home/Away indicator
    df['is_home'] = df['was_home'].astype(int)

    # 2. Season progression features
    # Season stage: early (GW 1-13), mid (GW 14-26), late (GW 27-38)
    df['season_stage'] = pd.cut(df['GW'], bins=[0, 13, 26, 38],
                                 labels=['early', 'mid', 'late'])

    # Gameweek as percentage of season completion
    df['season_progress'] = df['GW'] / 38.0

    # 3. Price change features (if value column exists)
    if 'value' in df.columns:
        # Sort by player and time
        df = df.sort_values(['element', 'season', 'GW']).reset_index(drop=True)

        # Price change from previous gameweek
        df['price_change'] = df.groupby(['element', 'season'])['value'].diff()

        # Cumulative price change within season (from starting price)
        df['price_change_cumulative'] = df.groupby(['element', 'season'])['value'].transform(
            lambda x: x - x.iloc[0] if len(x) > 0 else 0
        )

        # Price trend: increasing (1), stable (0), decreasing (-1)
        df['price_trend'] = df['price_change'].apply(
            lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
        )
    return df

### Availability

Whether the player will be on the pitch at all -- the single biggest driver of
FPL points, and something the model previously had no feature for beyond raw
lagged minutes.

Played and started rates over 3, 5 and 10 matches, minutes volatility, blank
and start streaks, matches since a full appearance, and three role flags.
Built from minutes rather than `starts`, which does not exist before 2023-24.

Worth +0.003 R2 marginally. Real, but small: most of what it describes is
already in the lagged minutes columns.

In [ ]:
def add_availability_features(df):
    """Will this player be on the pitch at all?

    FPL's own xP beats these models by 0.03-0.15 R2, and roughly half of that
    edge is simply knowing the starting XI: in 2024-25, players who did not
    play carried a mean xP of 0.145 against 2.299 for players who did, and xP's
    R2 falls from 0.42 to 0.20 once you look only at players who appeared.

    Minutes are the single biggest driver of FPL points, and the model had no
    feature describing whether a player is nailed on, rotated, or frozen out --
    only raw lagged minutes. These reconstruct that from appearance history.

    Every feature is shifted by one match inside the player group, so a row
    only ever sees matches that had already been played. `minutes` is used
    rather than `starts` because starts does not exist before 2023-24, and
    restricting to it would throw away seven seasons.
    """
    df = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)

    appeared = (df['minutes'] > 0).astype(float)
    started = (df['minutes'] >= 60).astype(float)
    grouped = df.groupby('name')

    def prior_mean(series, window):
        """Mean over the previous `window` matches, excluding this one."""
        return series.groupby(df['name']).transform(
            lambda x: x.rolling(window=window, min_periods=1).mean().shift(1)
        )

    # How often they have been on the pitch lately.
    for window in (3, 5, 10):
        df[f'avail_played_rate_{window}'] = prior_mean(appeared, window)
        df[f'avail_started_rate_{window}'] = prior_mean(started, window)

    # Erratic minutes are the rotation signal; a nailed starter has low variance.
    df['avail_minutes_std_5'] = grouped['minutes'].transform(
        lambda x: x.rolling(window=5, min_periods=2).std().shift(1)
    )

    # Recent minutes against the longer baseline: positive means working back
    # into the side, negative means dropping out of it.
    df['avail_minutes_trend'] = (
        grouped['minutes'].transform(
            lambda x: x.rolling(window=3, min_periods=1).mean().shift(1))
        - grouped['minutes'].transform(
            lambda x: x.rolling(window=10, min_periods=1).mean().shift(1))
    )

    def run_length(flag):
        """Length of the current unbroken run of `flag`, up to the last match."""
        def _run(x):
            blocks = (~x.astype(bool)).cumsum()
            return x.groupby(blocks).cumsum().shift(1)
        return flag.groupby(df['name']).transform(_run)

    # Consecutive blanks is the closest thing here to an injury flag.
    df['avail_zero_streak'] = run_length(1.0 - appeared)
    df['avail_start_streak'] = run_length(started)

    # Matches since they last completed an hour.
    since = grouped['minutes'].transform(
        lambda x: x.ge(60).astype(int).shift(1).fillna(0)
    )
    df['avail_games_since_start'] = since.groupby(df['name']).transform(
        lambda x: x.groupby(x.cumsum()).cumcount()
    )

    # Share of this season's matches they have featured in so far. Reset per
    # season, because last season's role says little after a transfer.
    df['avail_season_played_rate'] = appeared.groupby(
        [df['name'], df['season']]
    ).transform(lambda x: x.expanding().mean().shift(1))

    # Three coarse roles, which the linear models can use directly.
    rate5 = df['avail_started_rate_5']
    played5 = df['avail_played_rate_5']
    df['avail_is_nailed'] = (rate5 >= 0.8).astype(int)
    df['avail_is_rotation_risk'] = ((played5 >= 0.2) & (played5 < 0.8)).astype(int)
    df['avail_is_frozen_out'] = (played5 < 0.2).astype(int)

    # Days of rest. Long gaps follow injuries; short ones mean congestion.
    if 'kickoff_time' in df.columns:
        kickoff = pd.to_datetime(df['kickoff_time'], errors='coerce', utc=True)
        df['avail_days_since_last'] = (
            kickoff - kickoff.groupby(df['name']).shift(1)
        ).dt.total_seconds() / 86400.0

    created = [c for c in df.columns if c.startswith('avail_')]
    print(f"Added {len(created)} availability features:")
    for c in created:
        filled = df[c].notna().mean() * 100
        print(f"  {c:<32} {filled:5.1f}% populated")
    return df


print("Availability feature function defined!")

### Expected goals

Chances created rather than whether they went in. A striker with 0.8 xG and no
goal is a better bet next week than one who scored from his only touch.

Measured honestly, this family adds **nothing**: 0.282 R2 on its own but
-0.000 on top of the other features. Its signal is already carried by rolling
`total_points`, `bps` and `ict_index`. It is kept because it costs nothing to
compute and a two-stage model may yet separate it, not because it currently
helps.

xG begins in 2022-23, so `has_xg` marks the rows that predate it -- otherwise
a model reads eight seasons of zeros as "took no shots" rather than "not
recorded".

In [ ]:
def add_expected_features(df):
    """Lagged and rolling xG / xA.

    Diagnostics put the model's R2 among players who actually appeared at
    0.052: conditional on playing, it barely beat the mean. Expected goals and
    assists target exactly that gap. They measure the chances a player got
    rather than whether they went in, so they carry the signal a scoreline
    throws away -- a striker with 0.8 xG and no goal is a better bet next week
    than one who scored from his only touch.

    Named xg_* so the feature selector picks the family up by prefix, and
    shifted a match inside the player group like every other feature here.
    """
    source = [c for c in ('expected_goals', 'expected_assists',
                          'expected_goal_involvements', 'expected_goals_conceded')
              if c in df.columns]
    if not source:
        print("No expected-goals columns present; skipping.")
        return df

    df = df.sort_values(['name', 'season', 'game_number']).reset_index(drop=True)
    grouped = df.groupby('name')

    for col in source:
        short = col.replace('expected_', 'x').replace('goal_involvements', 'gi') \
                   .replace('goals_conceded', 'gc').replace('goals', 'g') \
                   .replace('assists', 'a')
        values = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

        for lag in (1, 2, 3):
            df[f'xg_{short}_prev_{lag}'] = values.groupby(df['name']).shift(lag)
        for window in (3, 5, 10):
            df[f'xg_{short}_rolling_{window}'] = values.groupby(df['name']).transform(
                lambda x: x.rolling(window=window, min_periods=1).mean().shift(1))

    # Overperformance: goals scored against the chances taken. A large positive
    # run is usually finishing luck rather than skill, and tends to come back.
    if {'goals_scored', 'expected_goals'} <= set(df.columns):
        goals = grouped['goals_scored'].transform(
            lambda x: x.rolling(window=10, min_periods=1).sum().shift(1))
        xg = grouped['expected_goals'].transform(
            lambda x: x.rolling(window=10, min_periods=1).sum().shift(1))
        df['xg_overperformance_10'] = goals - xg

    # Whether this row's season predates the stat at all. Without it a model
    # reads the zeros of 2016-17 as "took no shots" rather than "not recorded".
    if 'has_xg' in df.columns:
        df['xg_is_recorded'] = df['has_xg'].astype(int)

    created = [c for c in df.columns if c.startswith('xg_')]
    print(f"Added {len(created)} expected-goals features:")
    for c in created:
        nonzero = (df[c].fillna(0) != 0).mean() * 100
        print(f"  {c:<30} {nonzero:5.1f}% non-zero")
    return df


print("Expected-goals feature function defined!")

### Fixture difficulty

The most valuable family in the set, at +0.007 R2 marginally.

An earlier version of these features scored -0.009 alone. It mixed a team's
whole-season record with its recent form, ignored home and away entirely, and
measured the opponent's aggregate points rather than the goals a player
actually needs. These replace it with rolling attack and defence form split by
venue, and the gap between this side's attack and the opposition's defence.

In [ ]:
def add_fixture_features(df):
    """Opponent strength, measured the way a player actually needs it.

    Ablation put the previous opponent_* family at -0.009 R2 alone and +0.001
    marginally: seventeen features carrying no signal. FPL's own xP, which uses
    fixture difficulty properly, reaches 0.20 conditional R2 against this
    model's 0.052, so difficulty is not the problem -- the encoding was.

    Three things the old features got wrong:

      1. They mixed a team's whole-season record with its recent form, so a
         side that started badly and improved looked average all year.
      2. They ignored home and away, which is most of the effect being
         claimed -- conceding at home and conceding away are different rates.
      3. They were built from the opponent's aggregate points rather than
         from what a player in this position actually needs to know: how many
         goals that opponent concedes, and how often they keep a clean sheet.

    Everything below is a rolling mean over matches already played, shifted by
    one, computed per team and split by venue.
    """
    df = df.sort_values(['team', 'season', 'game_number']).reset_index(drop=True)

    # One row per team-match: what that team did in that fixture.
    # my_team_score / opponent_team_score are set in the cell that flips
    # team_h_score and team_a_score by venue, so they are the goals for and
    # against in this fixture regardless of which side the player was on.
    needed = {'my_team_score', 'opponent_team_score'}
    if not needed <= set(df.columns):
        print(f"missing {sorted(needed - set(df.columns))}; fixture features skipped")
        return df

    team_match = (df.groupby(['season', 'team', 'game_number', 'was_home'])
                    .agg(scored=('my_team_score', 'max'),
                         conceded=('opponent_team_score', 'max'))
                    .reset_index())
    if team_match['scored'].isna().all():
        print("team goal columns are empty; fixture features skipped")
        return df

    team_match = team_match.sort_values(['season', 'team', 'game_number'])
    grp = team_match.groupby(['season', 'team'])

    for window in (4, 8):
        team_match[f'att_form_{window}'] = grp['scored'].transform(
            lambda x: x.rolling(window, min_periods=1).mean().shift(1))
        team_match[f'def_form_{window}'] = grp['conceded'].transform(
            lambda x: x.rolling(window, min_periods=1).mean().shift(1))
        team_match[f'cs_rate_{window}'] = grp['conceded'].transform(
            lambda x: x.eq(0).rolling(window, min_periods=1).mean().shift(1))

    # Same three, split by venue: a team's away defence is its own statistic.
    venue = team_match.groupby(['season', 'team', 'was_home'])
    team_match['att_form_venue'] = venue['scored'].transform(
        lambda x: x.rolling(6, min_periods=1).mean().shift(1))
    team_match['def_form_venue'] = venue['conceded'].transform(
        lambda x: x.rolling(6, min_periods=1).mean().shift(1))

    form_cols = [c for c in team_match.columns
                 if c.startswith(('att_form', 'def_form', 'cs_rate'))]

    # Attach the player's own team form...
    own = team_match[['season', 'team', 'game_number', 'was_home'] + form_cols]
    own = own.rename(columns={c: f'fx_own_{c}' for c in form_cols})
    df = df.merge(own, on=['season', 'team', 'game_number', 'was_home'], how='left')

    # ...and the opponent's, taken from the opponent's own row in that match,
    # which is the same game_number with the venue flipped.
    opp = team_match[['season', 'team', 'game_number', 'was_home'] + form_cols].copy()
    opp['was_home'] = ~opp['was_home'].astype(bool)
    opp = opp.rename(columns={'team': 'opponent_team',
                              **{c: f'fx_opp_{c}' for c in form_cols}})
    df = df.merge(opp, on=['season', 'opponent_team', 'game_number', 'was_home'],
                  how='left')

    # What a player actually wants: the gap between his side's attack and the
    # opposition's defence, and vice versa.
    if {'fx_own_att_form_8', 'fx_opp_def_form_8'} <= set(df.columns):
        df['fx_attack_edge'] = df['fx_own_att_form_8'] - df['fx_opp_def_form_8']
        df['fx_defence_edge'] = df['fx_opp_att_form_8'] - df['fx_own_def_form_8']
        df['fx_cs_chance'] = df['fx_own_cs_rate_8'] - df['fx_opp_att_form_8']

    df['fx_is_home'] = df['was_home'].astype(int)

    created = [c for c in df.columns if c.startswith('fx_')]
    print(f"Added {len(created)} fixture features:")
    for c in created:
        print(f"  {c:<32} {df[c].notna().mean() * 100:5.1f}% populated")
    return df


print("Fixture feature function defined!")

### Building the features

One pass over the six families, in dependency order.

In [ ]:
# Each function takes a frame and returns it with one family of features
# added. They run in this order because later ones read columns the earlier
# ones create -- availability needs minutes history, and the fixture features
# need my_team_score, which the merge stage sets.
#
# Every one of them shifts its output by one match inside the player group, so
# no row can see the fixture it is describing.
FEATURE_STAGES = [
    ('previous-match statistics', add_previous_game_stats),
    ('rolling player form',       add_rolling_player_stats),
    ('season and price context',  add_context_features),
    ('availability',              add_availability_features),
    ('expected goals',            add_expected_features),
    ('fixture difficulty',        add_fixture_features),
]

featured = all_seasons_data
for label, build in FEATURE_STAGES:
    before = featured.shape[1]
    print(f"\n=== {label} ===")
    featured = build(featured)
    print(f"  {featured.shape[1] - before} columns added "
          f"({featured.shape[1]} total)")

all_seasons_data_featured = featured

print(f"\nFinal shape: {all_seasons_data_featured.shape[0]:,} rows "
      f"x {all_seasons_data_featured.shape[1]} columns")
print(f"Features engineered: "
      f"{all_seasons_data_featured.shape[1] - all_seasons_data.shape[1]}")

In [ ]:
# save the final dataset with features
all_seasons_data_featured.to_csv('all_seasons_data_featured.csv', index=False)

## 4. Split, preprocess and train

The order here matters more than the choice of model.

**Split by season, never at random.** This is panel data: one row per player
per gameweek, and the features are rolling means of recent matches. A random
split puts GW12 in train and GW13 of the same season in test, where the two
share almost all of their information. Every score produced that way is
optimistic. Whole seasons are held out instead, which asks the question that
matters: given everything up to now, how well do we predict a season nobody
has seen?

**Fit every transform on the training fold alone.** Scalers, encoders and the
hyperparameter search all see training seasons only. Fitting a scaler on the
full frame leaks the test distribution into training through the mean and
variance.

**One model per position.** A goalkeeper earns points from saves and clean
sheets, a forward from goals. Pooling them forces one set of coefficients to
describe four different scoring systems.

In [ ]:
import pandas as pd
# Written with encoding='latin-1' by older versions of this pipeline;
# pandas defaults to utf-8 on read, so fall back rather than crash on the
# first accented player name.
try:
    all_seasons_data_featured = pd.read_csv('all_seasons_data_featured.csv')
except UnicodeDecodeError:
    all_seasons_data_featured = pd.read_csv('all_seasons_data_featured.csv',
                                            encoding='latin-1')

In [ ]:
# What feature engineering produced, before anything is selected.
print(f"{all_seasons_data_featured.shape[0]:,} rows "
      f"x {all_seasons_data_featured.shape[1]} columns")
print()

families = {
    'lagged (_prev_)':       lambda c: '_prev_' in c,
    'rolling (_rolling_)':   lambda c: '_rolling_' in c,
    'availability (avail_)': lambda c: c.startswith('avail_'),
    'expected goals (xg_)':  lambda c: c.startswith('xg_'),
    'fixture (fx_)':         lambda c: c.startswith('fx_'),
}
for label, matches in families.items():
    count = sum(1 for c in all_seasons_data_featured.columns if matches(c))
    print(f"  {label:<24}{count:>4}")

# Anything non-numeric must be encoded or excluded before it reaches a model,
# so it is worth seeing the list rather than discovering it in a stack trace.
non_numeric = [c for c in all_seasons_data_featured.columns
               if not pd.api.types.is_numeric_dtype(all_seasons_data_featured[c])]
print()
print(f"  non-numeric ({len(non_numeric)}): {non_numeric}")

### Preparing the feature matrix

Four things have to happen before a model sees this frame: drop the columns
that describe the match being predicted, encode what is not numeric, choose
the feature set, and decide what to do about missing values.

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Create a copy to work with
df = all_seasons_data_featured.copy()

print(f"Initial dataset shape: {df.shape}")
print(f"Total missing values: {df.isnull().sum().sum()}")

### Excluding the present

The single most important cell in the notebook. Every column below describes
the match being predicted -- goals scored *in* it, minutes played *in* it,
bonus awarded *for* it. None is knowable at the point a team is picked, and
leaving any of them in produces a model that looks excellent and is worthless.

The lagged and rolling versions of these same statistics stay, because they
describe matches already played.

In [ ]:
# Columns to exclude from features (these are current gameweek stats or identifiers)
# These features contain information about the CURRENT gameweek, not previous ones
exclude_from_training = [
    'total_points',  # Target variable
    'name',  # Identifier
    'element',  # Player ID
    'fixture',  # Match ID
    'kickoff_time',  # Time info
    'round',  # Round number
    'GW',  # Gameweek number
    'game_number',  # Game number
    'match_number',  # Match number

    # Current gameweek performance stats (not available at prediction time)
    'assists',  # Current GW assists
    'bonus',  # Current GW bonus
    'bps',  # Current GW BPS
    'clean_sheets',  # Current GW clean sheets
    'clearances_blocks_interceptions',  # Current GW defensive stats
    'creativity',  # Current GW creativity
    'goals_conceded',  # Current GW goals conceded
    'goals_scored',  # Current GW goals scored
    'ict_index',  # Current GW ICT
    'influence',  # Current GW influence
    'minutes',  # Current GW minutes played
    'own_goals',  # Current GW own goals
    'penalties_missed',  # Current GW penalties missed
    'penalties_saved',  # Current GW penalties saved
    'recoveries',  # Current GW recoveries
    'red_cards',  # Current GW red cards
    'saves',  # Current GW saves
    'selected',  # Current GW selection
    'tackles',  # Current GW tackles
    'team_a_score',  # Current GW away score
    'team_h_score',  # Current GW home score
    'threat',  # Current GW threat
    'transfers_balance',  # Current GW transfers
    'transfers_in',  # Current GW transfers in
    'transfers_out',  # Current GW transfers out
    'yellow_cards',  # Current GW yellow cards
    'defensive_contribution',  # Current GW defensive contribution
    'result',
    'my_team_score',
    'opponent_team_score'
]

# Verify all these columns exist
exclude_from_training = [col for col in exclude_from_training if col in df.columns]

print(f"Excluding {len(exclude_from_training)} columns that represent current gameweek data:")
print(exclude_from_training)

### Encoding the categoricals

Team, opponent and position are strings. Label encoding is enough here: the
tree models do not care about the implied ordering, and the linear models get
these columns dropped by the compact feature set anyway.

In [ ]:
# Get non-numeric columns (excluding those already in exclude list)
non_numeric_cols = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
print("Non-numeric columns found:")
print(non_numeric_cols)

# Create encoders dictionary to store all label encoders
label_encoders = {}

# Encode categorical features
for col in non_numeric_cols:
    if col not in exclude_from_training:
        print(f"\nEncoding {col}...")
        le = LabelEncoder()
        # Handle NaN values
        df[col] = df[col].fillna('missing')
        df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le
        print(f"  - Created {col}_encoded with {len(le.classes_)} unique values")

# For was_home (boolean), convert to int if not already
if 'was_home' in df.columns and df['was_home'].dtype == 'bool':
    df['was_home'] = df['was_home'].astype(int)

# For is_home (boolean), convert to int if not already
if 'is_home' in df.columns and df['is_home'].dtype == 'bool':
    df['is_home'] = df['is_home'].astype(int)

# Handle was_home_prev columns (convert True/False strings to 0/1)
was_home_prev_cols = [col for col in df.columns if 'was_home_prev_' in col]
for col in was_home_prev_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].map({'True': 1, 'False': 0, True: 1, False: 0})
        df[col] = df[col].fillna(0).astype(float)

print(f"\nEncoding complete. Created {len(label_encoders)} encoded features.")

### Choosing the feature set

Two sets are available. The compact set is the default, and is what the models
are trained on; `FPL_FEATURE_SET=full` uses everything.

Measurement drove this. Ablation showed no single feature family worth more
than +0.005 R2 on top of the others, while minutes history alone reached 0.326
against the full model's 0.339 -- the signal is heavily duplicated across
correlated columns. Keeping the families that carry it, plus the fixture
features, costs nothing and trains in a fraction of the time.

In [ ]:
# Anything not explicitly withheld, and not a raw string column, is a
# candidate. The encoded copies of team/opponent/position are kept; the
# originals are dropped, since a model cannot read "Arsenal".
candidates = [c for c in df.columns if c not in exclude_from_training]

original_non_numeric = [c for c in non_numeric_cols if c not in exclude_from_training]
training_features = [
    c for c in candidates
    if pd.api.types.is_numeric_dtype(df[c]) and c not in original_non_numeric
]

# Grouped by prefix, because that is how the ablation measures them and how
# the compact set selects them. A family that turns out to be worth nothing
# should be visible here as a row, not buried among two hundred column names.
FAMILIES = [
    ('previous match (_prev_)',  lambda c: '_prev_' in c),
    ('rolling form (_rolling_)', lambda c: '_rolling_' in c),
    ('availability (avail_)',    lambda c: c.startswith('avail_')),
    ('expected goals (xg_)',     lambda c: c.startswith('xg_')),
    ('fixture (fx_)',            lambda c: c.startswith('fx_')),
    ('encoded categorical',      lambda c: c.endswith('_encoded')),
]

print(f"{len(training_features)} candidate features\n")
claimed = set()
for label, matches in FAMILIES:
    members = [c for c in training_features if matches(c)]
    claimed.update(members)
    print(f"  {label:<28}{len(members):>4}")

remainder = [c for c in training_features if c not in claimed]
print(f"  {'everything else':<28}{len(remainder):>4}")
print(f"\n  everything else: {remainder}")

### Missing values

Rows below game_number 5 have too little history for a rolling mean and are
dropped outright. Beyond that, a missing lag means the match did not happen,
so zero is the honest fill -- with the caveat that `has_fbref_defensive` and
`has_xg` exist precisely so a model can tell that kind of zero from a real
one.

In [ ]:
# Check missing values in training features
print("Missing values in training features:")
missing_counts = df[training_features].isnull().sum()
features_with_missing = missing_counts[missing_counts > 0].sort_values(ascending=False)
print(f"\nFeatures with missing values: {len(features_with_missing)}")
if len(features_with_missing) > 0:
    print("\nTop 20 features with most missing values:")
    print(features_with_missing.head(20))

# Drop rows with game_number < 5
print(f"\nOriginal dataset shape: {df.shape}")
if 'game_number' in df.columns:
    rows_before = len(df)
    df = df[df['game_number'] >= 5]
    rows_dropped = rows_before - len(df)
    print(f"Dropped {rows_dropped} rows with game_number < 5")
    print(f"Dataset shape after filtering game_number >= 5: {df.shape}")
else:
    print("Warning: 'game_number' column not found in dataframe")

# Drop rows with any NaN/null values in training features or target
print("\nDropping rows with missing values...")
rows_before = len(df)
df = df.dropna(subset=training_features + ['total_points'])
rows_dropped = rows_before - len(df)
print(f"Dropped {rows_dropped} rows with missing values")

print(f"\nAfter cleaning:")
print(f"  Dataset shape: {df.shape}")
print(f"  Missing values in features: {df[training_features].isnull().sum().sum()}")
print(f"  Missing values in target: {df['total_points'].isnull().sum()}")

### Feature matrix and target

In [ ]:
import numpy as np
# Prepare feature matrix X and target vector y
X = df[training_features].copy()
y = df['total_points'].copy()

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"\nTarget variable statistics:")
print(y.describe())

# Check for any remaining infinite values
print(f"\nInfinite values in X: {np.isinf(X.select_dtypes(include=[np.number])).sum().sum()}")

# Replace any infinite values with NaN then fill with median
if np.isinf(X.select_dtypes(include=[np.number])).sum().sum() > 0:
    print("Replacing infinite values with median...")
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        if X[col].isnull().sum() > 0:
            X[col] = X[col].fillna(X[col].median())

print(f"\nFinal feature check:")
print(f"  X shape: {X.shape}")
print(f"  X missing values: {X.isnull().sum().sum()}")
print(f"  X infinite values: {np.isinf(X.select_dtypes(include=[np.number])).sum().sum()}")

### Standardisation

### The temporal split

Every split below is by whole season, never shuffled. See the next cell for why:
a random split on player-gameweek rows leaks neighbouring gameweeks across the
train/test boundary, because the features are lags and rolling averages of the
same recent matches.

| fold | seasons |
| --- | --- |
| train | 2016-17 .. 2022-23 |
| validation | 2023-24 |
| test | 2024-25, 2025-26 |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Temporal splitting helpers
# ─────────────────────────────────────────────────────────────────────────────
# This is panel time-series data: one row per player per gameweek. A random
# train_test_split puts GW12 of a season into train and GW13 of the SAME season
# into test. Those two rows share almost all of their information, because the
# features are lags and rolling averages of the same recent matches -- so the
# model is scored on weeks it has effectively already seen. Every R2 produced
# that way is optimistic.
#
# Splitting on whole seasons instead answers the question we actually care
# about: given everything up to now, how well do we predict a season we have
# never seen?

SEASON_ORDER = [
    '2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
    '2021-22', '2022-23', '2023-24', '2024-25', '2025-26',
]

# Two split regimes. The default uses every season available.
#
# "xg" exists because expected goals only start in 2022-23. Training on the
# full history with xG present means 84% of training rows carry a fabricated
# zero and the test season carries real values -- a distribution shift, and
# measurably harmful: xG scores 0.242 R2 on its own but -0.0015 marginally
# under the default split. Confining train, validation and test to the seasons
# that actually record it costs a lot of rows and removes the mismatch.
#
# Set FPL_SPLIT=xg to use it.
import os as _os_split

_SPLITS = {
    'full': (['2016-17', '2017-18', '2018-19', '2019-20', '2020-21',
              '2021-22', '2022-23'], ['2023-24'], ['2024-25', '2025-26']),
    'xg':   (['2022-23', '2023-24'], ['2024-25'], ['2025-26']),
}
SPLIT_NAME = _os_split.environ.get('FPL_SPLIT', 'full').lower()
if SPLIT_NAME not in _SPLITS:
    raise ValueError(f"FPL_SPLIT must be one of {sorted(_SPLITS)}, got {SPLIT_NAME!r}")
TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS = _SPLITS[SPLIT_NAME]
print(f"split: {SPLIT_NAME.upper()}  train={TRAIN_SEASONS} val={VAL_SEASONS} test={TEST_SEASONS}")

# Number of folds for the inner cross-validation used during hyperparameter
# search. TimeSeriesSplit, not KFold: the inner CV has to respect time order
# too, otherwise the leak just moves from the outer split to the inner one.
INNER_CV_SPLITS = 3


def temporal_masks(frame):
    """Boolean train/val/test masks for `frame`, split on whole seasons.

    Masks are aligned to the frame's own index, so they can be applied to
    anything sharing that index (X, y, ...).
    """
    if 'season' not in frame.columns:
        raise ValueError(
            "temporal_masks() needs a 'season' column. If you are passing a "
            "feature matrix, pass the row frame it came from instead."
        )

    season = frame['season'].astype(str)

    unknown = sorted(set(season.unique()) - set(SEASON_ORDER))
    if unknown:
        raise ValueError(f"unrecognised season(s) {unknown}; update SEASON_ORDER")

    return season.isin(TRAIN_SEASONS), season.isin(VAL_SEASONS), season.isin(TEST_SEASONS)


def chronological_order(frame):
    """Index of `frame` sorted by season then gameweek.

    Reordering rows into time order is what makes TimeSeriesSplit meaningful
    for the inner CV: fold k must be entirely earlier than fold k+1.
    """
    rank = {s: i for i, s in enumerate(SEASON_ORDER)}
    key = pd.DataFrame({
        '_season': frame['season'].astype(str).map(rank),
        '_gw': frame['GW'] if 'GW' in frame.columns else 0,
    }, index=frame.index)
    return key.sort_values(['_season', '_gw']).index


def describe_split(train_mask, val_mask, test_mask, label=''):
    """Print the size of each fold, and warn about rows that fell through."""
    n_train, n_val, n_test = int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum())
    total = len(train_mask)
    print(f"{label}Temporal split ({total} rows):")
    for name, n, seasons in (
        ('train', n_train, TRAIN_SEASONS),
        ('val', n_val, VAL_SEASONS),
        ('test', n_test, TEST_SEASONS),
    ):
        pct = 100 * n / total if total else 0.0
        span = seasons[0] if len(seasons) == 1 else f"{seasons[0]}..{seasons[-1]}"
        print(f"  {name:<5} {n:>7,} rows ({pct:4.1f}%)  {span}")

    leftover = total - n_train - n_val - n_test
    if leftover:
        print(f"  WARNING: {leftover:,} rows fell outside every fold")
    if min(n_train, n_val, n_test) == 0:
        raise ValueError(f"{label}a fold is empty -- check the season values in this frame")


print("Temporal split helpers defined.")
print(f"  train: {TRAIN_SEASONS[0]}..{TRAIN_SEASONS[-1]}  ({len(TRAIN_SEASONS)} seasons)")
print(f"  val:   {VAL_SEASONS}")
print(f"  test:  {TEST_SEASONS}")
print(f"  inner CV: TimeSeriesSplit(n_splits={INNER_CV_SPLITS})")

In [ ]:
# Standardise using statistics from the TRAINING seasons only.
#
# The previous version called scaler.fit_transform(X) on the whole dataset and
# only split afterwards, so the mean and variance of the test seasons were
# baked into every training row. A scaler must never see data it will later be
# evaluated on.

train_mask, val_mask, test_mask = temporal_masks(df.loc[X.index])
describe_split(train_mask, val_mask, test_mask)

scaler = StandardScaler()
scaler.fit(X.loc[train_mask])

X_scaled = pd.DataFrame(
    scaler.transform(X),
    columns=X.columns,
    index=X.index,
)

print("\nFeature standardization complete (fitted on training seasons only).")
print("\nTraining-fold statistics after scaling (should be ~0 mean, ~1 std):")
print(X_scaled.loc[train_mask].describe().loc[['mean', 'std', 'min', 'max']].iloc[:, :5])

print("\nTest-fold statistics after scaling (drift from 0/1 here is real, not a bug):")
print(X_scaled.loc[test_mask].describe().loc[['mean', 'std', 'min', 'max']].iloc[:, :5])

print(f"\nScaled features shape: {X_scaled.shape}")

### Applying the split

In [ ]:
# Apply the temporal split computed in the previous cell.
#
# Was: train_test_split(X_scaled, y, test_size=0.2, shuffle=True). On panel
# time-series data that leaks neighbouring gameweeks across the boundary; see
# the temporal split helper cell for the full reasoning.

X_train, y_train = X_scaled.loc[train_mask], y.loc[train_mask]
X_val, y_val = X_scaled.loc[val_mask], y.loc[val_mask]
X_test, y_test = X_scaled.loc[test_mask], y.loc[test_mask]

print("Data split complete (by season, no shuffling).\n")
for name, Xs, ys, seasons in (
    ('Training', X_train, y_train, TRAIN_SEASONS),
    ('Validation', X_val, y_val, VAL_SEASONS),
    ('Testing', X_test, y_test, TEST_SEASONS),
):
    print(f"{name} set  ({', '.join(seasons)}):")
    print(f"  X shape: {Xs.shape}")
    print(f"  y mean:  {ys.mean():.3f}   y std: {ys.std():.3f}")
    print(f"  share:   {100 * len(Xs) / len(X_scaled):.1f}% of {len(X_scaled)} rows\n")

# The target mean drifting between folds is expected and worth seeing: FPL
# scoring rules changed over these seasons, so a model trained on 2016-22 is
# predicting a slightly different game than 2024-26.

### What the models will actually see

In [ ]:
# What the models will be handed, and what was deliberately withheld.
FAMILIES = [
    ('previous match (_prev_)',  lambda c: '_prev_' in c),
    ('rolling form (_rolling_)', lambda c: '_rolling_' in c),
    ('availability (avail_)',    lambda c: c.startswith('avail_')),
    ('expected goals (xg_)',     lambda c: c.startswith('xg_')),
    ('fixture (fx_)',            lambda c: c.startswith('fx_')),
    ('encoded categorical',      lambda c: c.endswith('_encoded')),
]

print("=" * 70)
print(f"FEATURE SET: {FEATURE_SET.upper()}   ({len(training_features)} features)")
print("=" * 70)

accounted = set()
for label, matches in FAMILIES:
    members = [c for c in training_features if matches(c)]
    accounted.update(members)
    print(f"  {label:<28}{len(members):>4}")
other = [c for c in training_features if c not in accounted]
print(f"  {'other':<28}{len(other):>4}   {other[:6]}{' ...' if len(other) > 6 else ''}")

print(f"\nWithheld: {len(exclude_from_training)} columns describing the match")
print("being predicted -- its goals, minutes, bonus and bps. None is knowable")
print("when a team is picked, and any one of them would inflate every score.")

print(f"\nRows: {len(df):,}")
print(f"  train  {train_mask.sum():>7,}   {TRAIN_SEASONS[0]}..{TRAIN_SEASONS[-1]}")
print(f"  val    {val_mask.sum():>7,}   {', '.join(VAL_SEASONS)}")
print(f"  test   {test_mask.sum():>7,}   {', '.join(TEST_SEASONS)}")
print(f"\nTarget: mean {y.mean():.3f}, sd {y.std():.3f}, "
      f"range {int(y.min())} to {int(y.max())}")

### Model training

Four candidates per position -- Ridge, ElasticNet, XGBoost and LightGBM --
each tuned with `GridSearchCV` over a `TimeSeriesSplit`. The inner CV has to
respect time order too, otherwise the leak simply moves from the outer split
to the inner one.

In [ ]:
# Import regression models and evaluation metrics
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time


In [ ]:
from collections import defaultdict

def auto_group_features(columns):
    groups = defaultdict(list)

    for col in columns:
        base = col.split("_prev")[0]
        base = base.split("_rolling")[0]
        groups[base].append(col)

    return dict(groups)
# Group features based on common prefixes
feature_groups = auto_group_features(training_features)
print("Feature groups identified:")
for group, cols in feature_groups.items():
    print(f"  - {group}: {len(cols)} features")
# print the keys as list
print("Feature group keys:", list(feature_groups.keys()))


In [ ]:
# Availability features are named avail_* and have no _prev/_rolling
# suffix, so auto_group_features gives each its own single-member group.
# Collect them by prefix rather than listing eighteen .get() calls.
AVAILABILITY_FEATURES = sorted(c for c in training_features
                               if c.startswith('avail_'))
# Same prefix trick for the two new families.
EXPECTED_FEATURES = sorted(c for c in training_features
                          if c.startswith('xg_'))
FIXTURE_FEATURES = sorted(c for c in training_features
                         if c.startswith('fx_'))
print(f"Availability features picked up: {len(AVAILABILITY_FEATURES)}")
print(f"Expected-goals features picked up: {len(EXPECTED_FEATURES)}")
print(f"Fixture features picked up: {len(FIXTURE_FEATURES)}")


COMMON_FEATURES  = feature_groups.get('value', []) + \
    feature_groups.get('bonus', []) + \
    feature_groups.get('bps', []) + \
    feature_groups.get('clearances_blocks_interceptions', []) + \
    feature_groups.get('minutes', []) + \
    feature_groups.get('own_goals', []) + \
    feature_groups.get('red_cards', []) + \
    feature_groups.get('selected', []) + \
    feature_groups.get('total_points', []) + \
    feature_groups.get('transfers_balance', []) + \
    feature_groups.get('transfers_in', []) + \
    feature_groups.get('transfers_out', []) + \
    feature_groups.get('yellow_cards', []) + \
    feature_groups.get('overall_team_strength', []) + \
    feature_groups.get('team_total_points', []) + \
    feature_groups.get('opponent_overall_team_strength', []) + \
    feature_groups.get('opponent_team_total_points', []) + \
    feature_groups.get('offensive_advantage', []) + \
    feature_groups.get('defensive_advantage', []) + \
    feature_groups.get('overall_advantage', []) + \
    feature_groups.get('opponent_difficulty', []) + \
    feature_groups.get('is_home', []) + \
    feature_groups.get('price_change', []) + \
    feature_groups.get('price_change_cumulative', []) + \
    feature_groups.get('price_trend', []) + \
    feature_groups.get('my_team_score', []) + \
    feature_groups.get('opponent_team_score', []) + \
    feature_groups.get('result', []) + \
    AVAILABILITY_FEATURES + \
    EXPECTED_FEATURES + \
    FIXTURE_FEATURES




ONLY_NOT_GK_FEATURES = feature_groups.get('assists', []) + \
    feature_groups.get('creativity', []) + \
    feature_groups.get('goals_scored', []) + \
    feature_groups.get('ict_index', []) + \
    feature_groups.get('influence', []) + \
    feature_groups.get('penalties_missed', []) + \
    feature_groups.get('recoveries', []) + \
    feature_groups.get('tackles', []) + \
    feature_groups.get('threat', []) + \
    feature_groups.get('defensive_contribution', []) + \
    feature_groups.get('offensive_strength', []) + \
    feature_groups.get('team_goals_scored', []) + \
    feature_groups.get('opponent_defensive_strength', []) + \
    feature_groups.get('opponent_team_goals_conceded', [])



ONLY_NOT_FWD_FEATURES = feature_groups.get('clean_sheets', []) + \
    feature_groups.get('goals_conceded', []) + \
    feature_groups.get('defensive_strength', []) + \
    feature_groups.get('team_goals_conceded', []) + \
    feature_groups.get('team_clean_sheet', []) + \
    feature_groups.get('opponent_offensive_strength', []) + \
    feature_groups.get('opponent_team_goals_scored', []) + \
    feature_groups.get('opponent_team_clean_sheet', [])


ONLY_GPK_FEATURES = feature_groups.get('saves', []) + \
    feature_groups.get('penalties_saved', []) + \
    feature_groups.get('team_saves', [])

GK_FEATURES = COMMON_FEATURES + ONLY_GPK_FEATURES + ONLY_NOT_FWD_FEATURES
DEF_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES
MID_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES
FWD_FEATURES = COMMON_FEATURES + ONLY_NOT_GK_FEATURES + ONLY_NOT_FWD_FEATURES



POSITION_FEATURES = {
    'GK': GK_FEATURES,
    'DEF': DEF_FEATURES,
    'MID': MID_FEATURES,
    'FWD': FWD_FEATURES
}

# ---------------------------------------------------------------------------
# Feature-set size
#
# scripts/ablate.py measured every family two ways: what it adds on top of
# everything else, and what it scores on its own. No family is worth more than
# +0.005 marginally, while minutes history alone reaches 0.326 against the full
# model's 0.339 -- the 220 columns are largely restatements of each other.
#
# Keeping only the families that carry signal costs about 0.003 R2 and drops
# 180 columns. The families left out include the 17 opponent-strength and
# fixture-difficulty features, which score below the mean on their own, and the
# FBref defensive columns, worth -0.0001 marginally.
#
# The FBref *data* still matters and is not going anywhere: defensive
# contribution decides a +2 point bonus on 10,604 rows, so it is part of the
# target even though it is not worth much as a feature.
#
# Set FPL_FEATURE_SET=full in the environment to train on all 220 again.
# ---------------------------------------------------------------------------
import os as _os

COMPACT_PREFIXES = (
    'minutes', 'avail_', 'total_points_', 'bps', 'value', 'ict_index',
    # Added after measurement: the fixture features are worth
    # +0.0074 R2 marginally, the largest of any group, and 37x what the
    # opponent_* family they replaced managed (+0.0002).
    'fx_',
)

FEATURE_SET = _os.environ.get('FPL_FEATURE_SET', 'compact').lower()

if FEATURE_SET == 'compact':
    _full_counts = {pos: len(feats) for pos, feats in POSITION_FEATURES.items()}
    POSITION_FEATURES = {
        pos: [f for f in feats if f.startswith(COMPACT_PREFIXES)]
        for pos, feats in POSITION_FEATURES.items()
    }
    print(f"feature set: COMPACT {COMPACT_PREFIXES}")
    for pos, feats in POSITION_FEATURES.items():
        print(f"  {pos}: {_full_counts[pos]} -> {len(feats)} features")
    for pos, feats in POSITION_FEATURES.items():
        assert len(feats) >= 20, (
            f"{pos} kept only {len(feats)} features; the compact prefixes match "
            f"almost nothing, which means the feature names have changed"
        )
else:
    print(f"feature set: FULL ({FEATURE_SET!r})")

print("Position-specific feature sets defined!")
for pos, features in POSITION_FEATURES.items():
    print(f"{pos}: {len(features)} features")

### Per-position data

`prepare_position_data` raises if too few of the requested features are present
rather than quietly training on whatever it finds. That guard exists because
the original notebook silently trained every model on a single column, price,
and reported the result as a working model.

In [ ]:
# Prepare data for modeling

# If a position's feature list is largely absent from the frame we were handed,
# that is a bug in the caller, not something to work around. Silently training
# on whatever happened to survive is how the "direct" models ended up fitted on
# a single column ('value') while reporting themselves as 201-feature models.
MIN_FEATURE_COVERAGE = 0.90


def prepare_position_data(df, position, features, min_coverage=MIN_FEATURE_COVERAGE):
    """Prepare X, y for a single position.

    Raises if fewer than `min_coverage` of the requested features exist in
    `df`, which almost always means the wrong dataframe was passed in.
    """

    # Filter by position
    pos_df = df[df['position'] == position].copy()

    # Get available features (some may not exist)
    available_features = [f for f in features if f in pos_df.columns]
    missing_features = [f for f in features if f not in pos_df.columns]

    coverage = len(available_features) / len(features) if features else 0.0
    if coverage < min_coverage:
        preview = ', '.join(missing_features[:8])
        more = f" (+{len(missing_features) - 8} more)" if len(missing_features) > 8 else ""
        raise ValueError(
            f"{position}: only {len(available_features)}/{len(features)} requested "
            f"features exist in this dataframe ({coverage:.1%} < {min_coverage:.0%}).\n"
            f"  Missing: {preview}{more}\n"
            f"  This usually means an un-engineered dataframe was passed. The "
            f"position feature lists are built from the FEATURED frame ('df'), "
            f"not from 'all_seasons_data'."
        )

    if missing_features:
        print(f"  note: {len(missing_features)} of {len(features)} features absent, "
              f"training on {len(available_features)}")

    # Remove rows with missing target
    pos_df = pos_df.dropna(subset=['total_points'])

    # Fill missing features with 0
    for col in available_features:
        pos_df[col] = pos_df[col].fillna(0)

    # Remove infinite values
    pos_df = pos_df.replace([np.inf, -np.inf], 0)

    X = pos_df[available_features]
    y = pos_df['total_points']

    return X, y, available_features, pos_df


print("Data preparation function defined!")
print(f"  guard: raises if <{MIN_FEATURE_COVERAGE:.0%} of requested features are present")

### Search grids

The ranges have been re-centred twice. Every parameter initially came back
sitting on a grid edge, which means the optimum was outside the range being
searched. They are now positioned so the chosen value usually falls inside.

Three rounds of this moved test R2 by less than 0.005. The ceiling here is the
features, not the search.

In [ ]:
# Define hyperparameter grids for different models
PARAM_GRIDS = {
    'RandomForest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None]
    },
    'GradientBoosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'max_depth': [3, 5, 7, 9],
        'min_samples_split': [2, 5, 10],
        'subsample': [0.8, 0.9, 1.0]
    },
    'Ridge': {
        'alpha': [0.01, 0.1, 1, 10, 100]
    },
    'ElasticNet': {
        'alpha': [0.01, 0.1, 1],
        'l1_ratio': [0.2, 0.5, 0.8]
    }
}

PARAM_GRIDS['XGBoost'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

PARAM_GRIDS['LightGBM'] = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, -1],
    'num_leaves': [31, 50, 100],
    'subsample': [0.8, 0.9, 1.0]
}

print("Hyperparameter grids defined for models:")
for model_name in PARAM_GRIDS:
    print(f"  - {model_name}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
def tune_model(model, param_grid, X_train, y_train, cv=5):
    """Perform GridSearchCV for hyperparameter tuning"""

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    return grid_search.best_estimator_, grid_search.best_params_, grid_search.best_score_

def tune_model_randomized(model, param_distributions, X_train, y_train, n_iter=50, cv=5):
    """Perform RandomizedSearchCV for faster hyperparameter tuning"""

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )

    random_search.fit(X_train, y_train)

    return random_search.best_estimator_, random_search.best_params_, random_search.best_score_

print("Hyperparameter tuning functions defined!")

### Gradient boosting

In [ ]:
# Additional imports for model training
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
import xgboost as xgb
import lightgbm as lgb

print("Additional libraries imported successfully!")

### Training

In [ ]:
# Define the models to train
MODELS = {
    'Ridge': Ridge(),
    'ElasticNet': ElasticNet(max_iter=10000),
    #'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
    #'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
    'LightGBM': lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
}

# Reduced parameter grids for faster training (use full grids for production)
#
# Every hyperparameter in the previous grids came back sitting on an edge:
# Ridge chose alpha=10 (the maximum), and ElasticNet and XGBoost chose the
# minimum of every one of their ranges, in all eight fits. When the search
# always stops at the boundary, the grid is in the wrong place -- the optimum
# is somewhere outside it. Every range below is extended in the direction the
# search was pulling, which for the tree models means more regularisation.
#
# LightGBM previously had no grid at all (this entry was commented out), so it
# trained at defaults with unlimited leaf growth. It memorised the training
# seasons: mean train R2 0.577 against test 0.312, a gap of 0.26, where the
# tuned models sat at 0.00-0.04. num_leaves and min_child_samples are the two
# knobs that actually bound that.
PARAM_GRIDS_REDUCED = {
    'Ridge': {'alpha': [1, 10, 100, 1000]},
    'ElasticNet': {'alpha': [0.001, 0.01, 0.1], 'l1_ratio': [0.1, 0.3, 0.5]},
    #'RandomForest': {'n_estimators': [100, 200], 'max_depth': [10, 20], 'min_samples_split': [2, 5]},
    #'GradientBoosting': {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5]},
    'XGBoost': {'n_estimators': [200, 400], 'learning_rate': [0.02, 0.05], 'max_depth': [2, 3, 4]},
    'LightGBM': {'n_estimators': [200, 400], 'learning_rate': [0.02, 0.05], 'num_leaves': [15, 31],
                 'min_child_samples': [50]}
}

print("Models and reduced parameter grids defined!")
print(f"Models to train: {list(MODELS.keys())}")
for _name, _grid in PARAM_GRIDS_REDUCED.items():
    _combos = 1
    for _v in _grid.values():
        _combos *= len(_v)
    print(f"  {_name:<11} {_combos:>3} combinations x {INNER_CV_SPLITS} folds "
          f"= {_combos * INNER_CV_SPLITS} fits")

In [ ]:
from sklearn.model_selection import TimeSeriesSplit


def train_and_evaluate_models(df, positions, position_features, models, param_grids, use_tuning=False):

    results = {}

    for position in positions:
        print(f"\n{'='*80}")
        print(f"Training models for position: {position}")
        print(f"{'='*80}")

        # Prepare data for this position
        features = position_features[position]
        X, y, available_features, pos_df = prepare_position_data(df, position, features)

        print(f"Data shape: X={X.shape}, y={y.shape}")
        print(f"Available features: {len(available_features)}")

        if len(X) < 100:
            print(f"Skipping {position} - not enough data")
            continue

        # ── Temporal split ────────────────────────────────────────────────
        # Whole seasons, never shuffled. Rows are also put into time order so
        # the inner TimeSeriesSplit sees genuinely earlier -> later folds.
        order = chronological_order(pos_df)
        X, y = X.loc[order], y.loc[order]
        train_mask, val_mask, test_mask = (m.loc[order] for m in temporal_masks(pos_df))
        describe_split(train_mask, val_mask, test_mask, label=f"{position}: ")

        X_train, y_train = X[train_mask], y[train_mask]
        X_val, y_val = X[val_mask], y[val_mask]
        X_test, y_test = X[test_mask], y[test_mask]

        # Scale features -- fitted on the training fold only
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)

        position_results = {
            'scaler': scaler,
            'features': available_features,
            'X_test': X_test,
            'y_test': y_test,
            'split': {
                'train_seasons': TRAIN_SEASONS,
                'val_seasons': VAL_SEASONS,
                'test_seasons': TEST_SEASONS,
                'n_train': int(train_mask.sum()),
                'n_val': int(val_mask.sum()),
                'n_test': int(test_mask.sum()),
            },
            'models': {}
        }

        # Train each model
        for model_name, model in models.items():
            print(f"\n--- Training {model_name} ---")
            start_time = time.time()

            try:
                # Clone the model to avoid issues with refitting
                model_clone = type(model)(**model.get_params())

                if use_tuning and model_name in param_grids:
                    # Perform hyperparameter tuning.
                    # TimeSeriesSplit, not KFold: the training fold is in time
                    # order, and a shuffled inner CV would reintroduce exactly
                    # the leak the outer split just removed.
                    print(f"Tuning hyperparameters...")
                    grid_search = GridSearchCV(
                        model_clone, param_grids[model_name],
                        cv=TimeSeriesSplit(n_splits=INNER_CV_SPLITS),
                        scoring='neg_mean_squared_error', n_jobs=-1
                    )
                    grid_search.fit(X_train_scaled, y_train)
                    best_model = grid_search.best_estimator_
                    best_params = grid_search.best_params_
                    print(f"Best params: {best_params}")
                else:
                    # Train without tuning
                    best_model = model_clone
                    best_model.fit(X_train_scaled, y_train)
                    best_params = None

                # Predictions
                y_train_pred = best_model.predict(X_train_scaled)
                y_val_pred = best_model.predict(X_val_scaled)
                y_test_pred = best_model.predict(X_test_scaled)

                # Calculate metrics
                train_metrics = {
                    'r2': r2_score(y_train, y_train_pred),
                    'mae': mean_absolute_error(y_train, y_train_pred),
                    'mse': mean_squared_error(y_train, y_train_pred),
                    'rmse': np.sqrt(mean_squared_error(y_train, y_train_pred))
                }

                val_metrics = {
                    'r2': r2_score(y_val, y_val_pred),
                    'mae': mean_absolute_error(y_val, y_val_pred),
                    'mse': mean_squared_error(y_val, y_val_pred),
                    'rmse': np.sqrt(mean_squared_error(y_val, y_val_pred))
                }

                test_metrics = {
                    'r2': r2_score(y_test, y_test_pred),
                    'mae': mean_absolute_error(y_test, y_test_pred),
                    'mse': mean_squared_error(y_test, y_test_pred),
                    'rmse': np.sqrt(mean_squared_error(y_test, y_test_pred))
                }

                training_time = time.time() - start_time

                position_results['models'][model_name] = {
                    'model': best_model,
                    'best_params': best_params,
                    'train_metrics': train_metrics,
                    'val_metrics': val_metrics,
                    'test_metrics': test_metrics,
                    'training_time': training_time,
                    'y_test_pred': y_test_pred
                }

                print(f"Training R²: {train_metrics['r2']:.4f}, Validation R²: {val_metrics['r2']:.4f}, Test R²: {test_metrics['r2']:.4f}")
                print(f"Test MAE: {test_metrics['mae']:.4f}, Test RMSE: {test_metrics['rmse']:.4f}")
                print(f"Training time: {training_time:.2f}s")

            except Exception as e:
                print(f"Error training {model_name}: {str(e)}")
                continue

        results[position] = position_results

    return results

print("Training and evaluation function defined!")
print("  split: by season (temporal)   inner CV: TimeSeriesSplit")

### Results

In [ ]:
# Train models for all positions
# Set use_tuning=True for hyperparameter tuning (slower but potentially better results)
# Set use_tuning=False for faster training with default parameters

positions_to_train = ['GK', 'DEF', 'MID', 'FWD']

print("Starting model training for all positions...")
print("This may take a few minutes...\n")

# NOTE: `df` -- the FEATURED, encoded frame built earlier in this notebook.
#
# This used to pass `all_seasons_data`, which is the raw merged frame from
# BEFORE feature engineering. None of the lag/rolling/opponent columns in
# POSITION_FEATURES exist there, so prepare_position_data silently dropped all
# but one of them and every "direct" model was really a single-feature model
# fitted on 'value' (price). prepare_position_data now raises instead.
all_results = train_and_evaluate_models(
    df=df,
    positions=positions_to_train,
    position_features=POSITION_FEATURES,
    models=MODELS,
    param_grids=PARAM_GRIDS_REDUCED,
    use_tuning=True  # Set to True for hyperparameter tuning
)

print("\n" + "="*80)
print("Model training completed for all positions!")
print("="*80)

### Comparison

In [ ]:
def create_results_summary(results):
    """Create a summary DataFrame of all model results"""

    summary_data = []

    for position, pos_results in results.items():
        for model_name, model_results in pos_results['models'].items():
            summary_data.append({
                'Position': position,
                'Model': model_name,
                'Train_R2': model_results['train_metrics']['r2'],
                'Val_R2': model_results['val_metrics']['r2'],
                'Test_R2': model_results['test_metrics']['r2'],
                'Train_MAE': model_results['train_metrics']['mae'],
                'Val_MAE': model_results['val_metrics']['mae'],
                'Test_MAE': model_results['test_metrics']['mae'],
                'Train_RMSE': model_results['train_metrics']['rmse'],
                'Val_RMSE': model_results['val_metrics']['rmse'],
                'Test_RMSE': model_results['test_metrics']['rmse'],
                'Training_Time': model_results['training_time']
            })

    summary_df = pd.DataFrame(summary_data)
    return summary_df

# Create and display summary
summary_df = create_results_summary(all_results)
print("="*100)
print("MODEL PERFORMANCE SUMMARY - ALL POSITIONS")
print("="*100)
print(summary_df.to_string(index=False))

# Save summary to CSV
summary_df.to_csv('model_results_summary.csv', index=False)
print("\nResults saved to 'model_results_summary.csv'")

In [ ]:
# Display best model for each position
print("\n" + "="*80)
print("BEST MODEL FOR EACH POSITION (Based on Test R²)")
print("="*80)

for position in positions_to_train:
    if position in all_results:
        pos_models = all_results[position]['models']
        if pos_models:
            best_model_name = max(pos_models.keys(), key=lambda x: pos_models[x]['test_metrics']['r2'])
            best_r2 = pos_models[best_model_name]['test_metrics']['r2']
            best_mae = pos_models[best_model_name]['test_metrics']['mae']
            best_rmse = pos_models[best_model_name]['test_metrics']['rmse']
            print(f"\n{position}:")
            print(f"  Best Model: {best_model_name}")
            print(f"  Test R²: {best_r2:.4f}")
            print(f"  Test MAE: {best_mae:.4f}")
            print(f"  Test RMSE: {best_rmse:.4f}")

## 5. Two-stage models

A single regressor over every row spends its capacity on the easy half of
the problem: 60% of rows are players who did not appear. Measured, that
gave R2 0.35 overall but 0.076 among players who actually took the field.

Splitting it into P(plays) and E[points | plays] lifts the conditional
figure to 0.083 and improves every position. The play classifier reaches
0.954 AUC on its own, which is the part of the problem this project was
always solving well.

The haul classifier reaches 0.869 AUC but does **not** improve captain
ranking over expected points -- both catch about a third of hauls in the
top 5%. It is kept because the probability is more honest than an expected
value for a decision about upside, not because it scores better.

In [ ]:
class HurdleModel:
    """P(plays) x E[points | plays], fitted separately.

    Why bother. A single regressor over every row spends its capacity on the
    easy half of the problem: 60% of rows are players who did not appear and
    score almost exactly zero. Diagnostics showed the consequence -- overall R2
    0.333, but only 0.052 among players who actually took the field, and a
    predicted spread just 0.55x the real one. The model had learned to say
    "probably about one point" very accurately and could not pick a captain.

    Splitting it lets each half be trained on the question it is actually
    answering:

        stage 1   classifier   will this player be on the pitch at all?
        stage 2   regressor    given that he is, how many points?

    Stage 2 trains only on rows where the player appeared, so the near-zeros
    stop drowning the signal. The product is still an expected value, and it is
    still calibrated, but the two parts can now be inspected and improved
    separately -- and stage 1's probability is useful on its own.
    """

    def __init__(self, classifier=None, regressor=None, threshold_minutes=1):
        self.classifier = classifier
        self.regressor = regressor
        self.threshold_minutes = threshold_minutes
        self.scaler_c = StandardScaler()
        self.scaler_r = StandardScaler()

    def fit(self, X, y_points, minutes):
        played = (minutes >= self.threshold_minutes).astype(int)

        self.classifier.fit(self.scaler_c.fit_transform(X), played)

        mask = played.astype(bool)
        if mask.sum() < 50:
            raise ValueError(f"only {mask.sum()} rows with minutes; cannot fit stage 2")
        self.regressor.fit(self.scaler_r.fit_transform(X[mask]), y_points[mask])

        self.play_rate_ = float(played.mean())
        self.n_stage2_ = int(mask.sum())
        return self

    def predict_parts(self, X):
        p_play = self.classifier.predict_proba(self.scaler_c.transform(X))[:, 1]
        points_if_playing = self.regressor.predict(self.scaler_r.transform(X))
        # A negative score is possible in FPL but never the sensible forecast
        # for a player who starts; clipping keeps the product interpretable.
        return p_play, np.clip(points_if_playing, 0, None)

    def predict(self, X):
        p_play, conditional = self.predict_parts(X)
        return p_play * conditional


class HaulModel:
    """P(scoring 10 or more), which is what a captain pick actually needs.

    Expected points and upside come apart badly here. The regressor's ceiling
    prediction is around 8 while real returns reach 25, and its mean forecast
    for players who did haul was 3.17 against 1.17 for everyone else -- enough
    to rank, nowhere near enough to distinguish a captain from a steady four.

    Ranking by P(haul) instead asks the question directly. It is a rare event
    (1.7% of rows), so the probability is small everywhere; what matters is the
    ordering, and the lift over the base rate.
    """

    def __init__(self, classifier=None, haul_points=10):
        self.classifier = classifier
        self.haul_points = haul_points
        self.scaler = StandardScaler()

    def fit(self, X, y_points):
        hauled = (y_points >= self.haul_points).astype(int)
        if hauled.sum() < 30:
            raise ValueError(f"only {hauled.sum()} hauls in the training fold")
        self.classifier.fit(self.scaler.fit_transform(X), hauled)
        self.base_rate_ = float(hauled.mean())
        return self

    def predict_proba(self, X):
        return self.classifier.predict_proba(self.scaler.transform(X))[:, 1]


print("HurdleModel and HaulModel defined!")

## 6. Save models and artifacts

Everything `scripts/predict_gameweek.py` needs to score an unplayed
gameweek: the per-position model, its scaler, and the exact feature
list it was trained on.

In [ ]:
import joblib
import json
import os

SAVE_DIR = 'saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 80)
print("SAVING ALL MODELS & ARTIFACTS")
print("=" * 80)

# ──────────────────────────────────────────────
# 1. Save DIRECT per-position models (all_results)
# ──────────────────────────────────────────────
print("\n1. Saving direct per-position models from all_results ...")
direct_meta = {}

for position, pos_results in all_results.items():
    pos_dir = os.path.join(SAVE_DIR, 'direct', position)
    os.makedirs(pos_dir, exist_ok=True)

    # Save scaler
    joblib.dump(pos_results['scaler'], os.path.join(pos_dir, 'scaler.joblib'))

    # Save feature list
    with open(os.path.join(pos_dir, 'features.json'), 'w') as f:
        json.dump(pos_results['features'], f)

    # Save each trained model
    best_model_name = None
    best_r2 = -999
    for model_name, model_data in pos_results['models'].items():
        joblib.dump(model_data['model'], os.path.join(pos_dir, f'{model_name}.joblib'))
        if model_data['test_metrics']['r2'] > best_r2:
            best_r2 = model_data['test_metrics']['r2']
            best_model_name = model_name

    direct_meta[position] = {
        'best_model': best_model_name,
        'best_test_r2': float(best_r2),
        'features_count': len(pos_results['features']),
        'models_saved': list(pos_results['models'].keys()),
    }
    print(f"   {position}: best={best_model_name} (R2={best_r2:.4f}), saved {len(pos_results['models'])} models")

with open(os.path.join(SAVE_DIR, 'direct', 'meta.json'), 'w') as f:
    json.dump(direct_meta, f, indent=2)

# ──────────────────────────────────────────────
# 2. Save PCA per-position models (pca_model_results)
# ──────────────────────────────────────────────
if 'pca_model_results' not in globals():
    print("\n2. skipped: the PCA branch was not trained in this run")
    pca_model_results = {}

print("\n2. Saving PCA per-position models from pca_model_results ...")
pca_meta = {}

for position, pos_results in pca_model_results.items():
    pos_dir = os.path.join(SAVE_DIR, 'pca_models', position)
    os.makedirs(pos_dir, exist_ok=True)

    best_model_name = None
    best_r2 = -999
    for model_name, model_data in pos_results['models'].items():
        joblib.dump(model_data['model'], os.path.join(pos_dir, f'{model_name}.joblib'))
        if model_data['test_metrics']['r2'] > best_r2:
            best_r2 = model_data['test_metrics']['r2']
            best_model_name = model_name

    pca_meta[position] = {
        'best_model': best_model_name,
        'best_test_r2': float(best_r2),
        'n_pca_components': int(pos_results['n_pca_components']),
        'models_saved': list(pos_results['models'].keys()),
    }
    print(f"   {position}: best={best_model_name} (R2={best_r2:.4f}), PCA dims={pos_results['n_pca_components']}")

if pca_meta:
    os.makedirs(os.path.join(SAVE_DIR, 'pca_models'), exist_ok=True)
    with open(os.path.join(SAVE_DIR, 'pca_models', 'meta.json'), 'w') as f:
        json.dump(pca_meta, f, indent=2)

# ──────────────────────────────────────────────
# 3. Save PCA artifacts (position_feature_selection)
# ──────────────────────────────────────────────
if 'position_feature_selection' not in globals():
    print("\n3. skipped: no PCA artifacts were produced in this run")
    position_feature_selection = {}

print("\n3. Saving PCA transformers & scalers from position_feature_selection ...")

for position, pfs in position_feature_selection.items():
    pos_dir = os.path.join(SAVE_DIR, 'pca_artifacts', position)
    os.makedirs(pos_dir, exist_ok=True)

    joblib.dump(pfs['pca_model'], os.path.join(pos_dir, 'pca_model.joblib'))
    joblib.dump(pfs['scaler'], os.path.join(pos_dir, 'pca_scaler.joblib'))

    with open(os.path.join(pos_dir, 'pca_config.json'), 'w') as f:
        json.dump({
            'original_features': pfs['original_features'],
            'features_after_correlation': pfs['features_after_correlation'],
            'removed_by_correlation': pfs['removed_by_correlation'],
            'pca_n_components': int(pfs['pca_n_components']),
            'pca_explained_variance': pfs['pca_explained_variance'].tolist(),
        }, f, indent=2)
    print(f"   {position}: PCA {len(pfs['original_features'])} -> {pfs['pca_n_components']} dims")

# ──────────────────────────────────────────────
# 4. Save Stat-Based Predictor
# ──────────────────────────────────────────────
print("\n4. Saving ImprovedStatBasedPredictor (stat_predictor) ...")
# The stat predictor is trained further down the notebook, so it is absent
# when only the position models have been run (e.g. scripts/train.py).
if 'stat_predictor' in globals():
    joblib.dump(stat_predictor, os.path.join(SAVE_DIR, 'stat_predictor.joblib'))
    print(f"   Saved stat_predictor with {len(stat_predictor.stat_models)} stat categories")
else:
    print("   skipped: stat_predictor was not trained in this run")

# ──────────────────────────────────────────────
# 5. Save configuration / feature definitions
# ──────────────────────────────────────────────
print("\n5. Saving configuration & feature definitions ...")

# STATS_TO_PREDICT / ALL_STATS_TO_PREDICT belong to the stat-predictor
# section further down, which has not run when only the position models
# were trained. Record the temporal split too: anyone reloading these
# models needs to know which seasons they have already seen.
config = {
    'training_features': training_features,
    'POSITION_FEATURES': POSITION_FEATURES,
    'STATS_TO_PREDICT': globals().get('STATS_TO_PREDICT'),
    'ALL_STATS_TO_PREDICT': globals().get('ALL_STATS_TO_PREDICT'),
    'exclude_from_training': exclude_from_training,
    'split': {
        'train_seasons': TRAIN_SEASONS,
        'val_seasons': VAL_SEASONS,
        'test_seasons': TEST_SEASONS,
    },
}
joblib.dump(config, os.path.join(SAVE_DIR, 'config.joblib'))
print(f"   training_features: {len(training_features)} features")
print(f"   POSITION_FEATURES: {', '.join(f'{k}={len(v)}' for k, v in POSITION_FEATURES.items())}")

# ──────────────────────────────────────────────
# 6. Save Label Encoders
# ──────────────────────────────────────────────
print("\n6. Saving label encoders ...")
joblib.dump(label_encoders, os.path.join(SAVE_DIR, 'label_encoders.joblib'))
print(f"   Saved {len(label_encoders)} label encoders")

# ──────────────────────────────────────────────
# Summary
# ──────────────────────────────────────────────
print("\n" + "=" * 80)
print("ALL ARTIFACTS SAVED SUCCESSFULLY")
print("=" * 80)
print(f"\nSave directory: {os.path.abspath(SAVE_DIR)}")
for root, dirs, files in os.walk(SAVE_DIR):
    level = root.replace(SAVE_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for file in files:
        size_kb = os.path.getsize(os.path.join(root, file)) / 1024
        print(f"{subindent}{file} ({size_kb:.1f} KB)")